# 사례 1 실습 — 건설현장 안전장비(PPE) 미착용 감지

**파일**: `01_yolo_day1_ppe_lab_v1.1.ipynb` · **환경**: Google Colab (T4 GPU) · **모델**: YOLO26n · **데이터**: Ultralytics Construction-PPE (AGPL-3.0)

현장 CCTV 사진에서 헬멧·조끼 같은 안전장비를 찾고, **장비를 안 쓴 사람이 있으면 경고**하는 시스템을 처음부터 끝까지 만들어 봅니다. 마지막 장에서 결과물을 모아 **포트폴리오 한 장**을 자동으로 만듭니다.

| 장 | 내용 | 결과물 |
|---|---|---|
| 1 | 설치 | — |
| 2 | 데이터 살펴보기: 클래스 11개와 분포 | 클래스 분포 그래프 |
| 3 | 라벨 직접 열어 보기: `none`은 무엇일까 | 라벨 시각화 |
| 4 | 학습 | 학습된 모델 `best.pt` |
| 5 | 결과 읽기: mAP와 클래스별 AP | `metrics_by_class.csv` |
| 6 | 판정 로직: 사람마다 헬멧이 있는가 | `violation_report.csv` |
| 7 | 실패 사례 보기 | `failure_cases.jpg` |
| 8 | 포트폴리오 한 장 만들기 | `portfolio_ppe.md`, 결과 이미지 |

> 시작 전에 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하세요.

## 1장. 설치

In [ ]:
!pip install -q ultralytics shapely lap

In [ ]:
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (CPU 모드 — 4장에서 EPOCHS와 IMGSZ를 줄이세요)")

## 2장. 데이터 살펴보기

YAML 이름만 넣으면 데이터셋(약 178MB)을 자동으로 내려받습니다.

In [ ]:
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from ultralytics.data.utils import check_det_dataset

data = check_det_dataset("construction-ppe.yaml")
ROOT = Path(data["train"]).parent.parent
NAMES = data["names"]
print("클래스 11개:", NAMES)
for split in ["train", "val", "test"]:
    n_img = len(list((ROOT / "images" / split).glob("*")))
    print(f"{split:5s} 이미지 {n_img}장")

In [ ]:
def read_labels(split):
    rows = []
    for p in sorted((ROOT / "labels" / split).glob("*.txt")):
        for line in p.read_text().splitlines():
            if line.strip():
                c, cx, cy, w, h = line.split()[:5]
                rows.append({"image": p.stem, "cls": NAMES[int(c)], "w": float(w), "h": float(h)})
    return pd.DataFrame(rows)

lab = read_labels("train")
counts = lab["cls"].value_counts()
plt.figure(figsize=(10, 4))
colors = ["#E0602D" if c.startswith("no_") or c == "none" else "#2E5E8C" for c in counts.index]
counts.plot.bar(color=colors)
plt.title("train boxes per class (orange = missing PPE)"); plt.ylabel("boxes"); plt.xticks(rotation=45); plt.tight_layout(); plt.show()
print(counts.to_string())

**읽어 볼 점**
- 가장 많은 `Person`과 가장 적은 `no_boots`의 차이가 20배가 넘습니다. 학습 데이터가 적은 클래스는 성능도 낮게 나오기 쉽습니다. 5장에서 확인합니다.
- `helmet`(착용)은 있는데 `no_vest`(조끼 미착용)는 없습니다. 조끼를 안 입은 사람은 어떻게 찾을까요? → 6장의 판정 로직

In [ ]:
# 물체 크기: 이미지 한 변 대비 박스 한 변(%) — 작은 장비일수록 찾기 어렵습니다.
lab["side_pct"] = np.sqrt(lab["w"] * lab["h"]) * 100
size_tbl = lab.groupby("cls")["side_pct"].mean().sort_values().round(1)
print(size_tbl.to_string())

## 3장. 라벨 직접 열어 보기 — `none`은 무엇일까

공식 문서는 `none`을 "일반적인 none 클래스"라고만 적고 정의를 주지 않습니다. 직접 열어서 어디에 붙어 있는지 봅니다.

In [ ]:
import random, matplotlib.patches as patches
from PIL import Image

COLORS = {"helmet": "lime", "vest": "yellow", "Person": "white", "none": "red", "no_helmet": "orangered",
          "gloves": "orange", "boots": "violet", "goggles": "magenta", "no_goggle": "purple", "no_gloves": "brown", "no_boots": "gray"}

def show_labels(stems, split="train", only=None):
    fig, axes = plt.subplots(1, len(stems), figsize=(6 * len(stems), 6))
    for ax, stem in zip(np.atleast_1d(axes), stems):
        img = next((ROOT / "images" / split).glob(stem + ".*"))
        im = Image.open(img).convert("RGB"); W, H = im.size
        ax.imshow(im)
        for line in (ROOT / "labels" / split / f"{stem}.txt").read_text().splitlines():
            if not line.strip(): continue
            c, cx, cy, w, h = map(float, line.split()[:5]); name = NAMES[int(c)]
            if only and name not in only: continue
            ax.add_patch(patches.Rectangle(((cx - w/2) * W, (cy - h/2) * H), w * W, h * H, fill=False, color=COLORS[name], lw=2))
            ax.text((cx - w/2) * W, (cy - h/2) * H - 4, name, color=COLORS[name], fontsize=9, backgroundcolor="black")
        ax.set_title(stem); ax.axis("off")
    plt.show()

random.seed(1)
with_none = sorted(lab.loc[lab["cls"] == "none", "image"].unique())
show_labels(random.sample(with_none, 3), only=["none", "Person", "vest", "helmet"])

**질문**: 빨간 `none` 박스는 상체에 붙어 있나요, 얼굴에 붙어 있나요? 사진마다 다르다면, 모델은 `none`을 무엇으로 배울까요?

> 현업에서는 이런 상황을 막으려고 **라벨 기준서**(클래스마다 "무엇을, 어디까지, 어떻게" 그리는지 정한 문서)를 먼저 씁니다. 포트폴리오에 "공개 데이터의 라벨 정의가 모호해 ○○로 해석했다"고 적으면 데이터를 직접 들여다봤다는 증거가 됩니다.

## 4장. 학습

사전학습된 `yolo26n.pt`에서 출발합니다. 데이터는 손대지 않고 기본값으로 먼저 돌려 기준점(baseline)을 만드는 것이 현업의 첫 단계입니다.

In [ ]:
from ultralytics import YOLO

EPOCHS = 30        # T4 기준 약 6~10분. CPU라면 3, 시간이 남으면 50
IMGSZ = 640        # CPU라면 320

model = YOLO("yolo26n.pt")
results = model.train(
    data="construction-ppe.yaml",
    epochs=EPOCHS, patience=10,
    imgsz=IMGSZ, batch=16,
    seed=0,
    project="runs_ppe", name="baseline", exist_ok=True,
)
RUN = Path(results.save_dir)
OUT = Path("portfolio_ppe"); OUT.mkdir(exist_ok=True)     # 포트폴리오 결과물을 모을 폴더
print("학습 결과:", RUN)

In [ ]:
# 학습 곡선 한 장: 손실이 내려가고 mAP가 올라가는지만 확인합니다.
Image.open(RUN / "results.png")

## 5장. 결과 읽기 — mAP와 클래스별 AP

- **mAP50**: 예측 박스가 정답과 절반 이상 겹치면(IoU ≥ 0.5) 맞은 것으로 치고 매긴 점수 (0~1)
- **mAP50-95**: 겹침 기준을 0.50부터 0.95까지 엄격하게 올려 가며 매긴 점수의 평균
- **클래스별 AP**: 위 점수를 클래스마다 따로 본 것. mAP는 이것들의 평균입니다.

평균 하나만 보면 약한 클래스가 가려집니다. 클래스별로 펼쳐 봅니다.

In [ ]:
best = YOLO(RUN / "weights" / "best.pt")
m = best.val(data="construction-ppe.yaml", imgsz=IMGSZ, nms=True, verbose=False)
print(f"전체  mAP50 {m.box.map50:.3f} · mAP50-95 {m.box.map:.3f}")

per_cls = pd.DataFrame({
    "class": [NAMES[int(i)] for i in m.box.ap_class_index],
    "train_boxes": [int(counts.get(NAMES[int(i)], 0)) for i in m.box.ap_class_index],
    "precision": m.box.p, "recall": m.box.r,
    "AP50": m.box.ap50, "AP50_95": m.box.ap,
}).sort_values("AP50").round(3)
per_cls.to_csv(OUT / "metrics_by_class.csv", index=False)     # 열 이름은 영문 (Excel 호환)
per_cls

In [ ]:
# 학습 데이터 양과 AP의 관계
plt.figure(figsize=(7, 5))
plt.scatter(per_cls["train_boxes"], per_cls["AP50"], color="#E0602D")
for _, r in per_cls.iterrows():
    plt.annotate(r["class"], (r["train_boxes"], r["AP50"]), fontsize=9, xytext=(4, 4), textcoords="offset points")
plt.xlabel("train boxes"); plt.ylabel("AP50 (val)"); plt.title("more data → higher AP?"); plt.grid(alpha=0.3)
plt.savefig(OUT / "ap_vs_data.png", dpi=120, bbox_inches="tight"); plt.show()

> **표 읽는 팁**: `precision 1.000 · recall 0.000`인 클래스는 "잘 맞혔다"가 아니라 **그 클래스를 하나도 예측하지 않았다**는 뜻입니다(예측이 없으면 틀린 예측도 없어 정밀도가 1로 표시됩니다). 학습을 더 오래 하거나 데이터를 늘리면 바뀝니다.

**질문**
1. AP가 가장 낮은 클래스 세 개는 무엇인가요? 2장의 개수·크기 표와 비교하면 이유가 보이나요?
2. 현장 안전에서 꼭 잡아야 하는 것은 `helmet`일까요, `no_helmet`일까요? 그 클래스의 recall은 몇인가요?

## 6장. 판정 로직 — 사람마다 헬멧이 있는가

모델의 출력은 "여기 헬멧, 저기 사람"이라는 박스 목록일 뿐입니다. **"이 사람은 헬멧을 안 썼다"**는 판정은 규칙으로 붙여야 합니다. 두 방법을 비교합니다.

| 방법 | 규칙 | 장점 | 약점 |
|---|---|---|---|
| A. 미착용 클래스 직접 사용 | `no_helmet` 박스가 있으면 경고 | 간단 | `no_helmet` 학습 데이터가 적음 |
| B. 사람 + 장비 조합 | `Person` 박스의 위쪽 1/3 안에 `helmet`이 없으면 경고 | 조끼(`no_vest` 없음)에도 같은 규칙 적용 가능 | 사람이 겹치면 헷갈림 |

In [ ]:
CONF = 0.25
cid = {v: k for k, v in NAMES.items()}

def inside(inner, outer, top_ratio=1.0):
    # inner 박스의 중심이 outer 박스의 위쪽 top_ratio 영역 안에 있는가
    cx, cy = (inner[0] + inner[2]) / 2, (inner[1] + inner[3]) / 2
    x1, y1, x2, y2 = outer
    return x1 <= cx <= x2 and y1 <= cy <= y1 + (y2 - y1) * top_ratio

def judge(result):
    b = result.boxes.xyxy.cpu().numpy(); c = result.boxes.cls.cpu().numpy().astype(int)
    persons = b[c == cid["Person"]]; helmets = b[c == cid["helmet"]]; vests = b[c == cid["vest"]]
    rows = []
    for k, p in enumerate(persons):
        has_helmet = any(inside(h, p, top_ratio=1/3) for h in helmets)
        has_vest = any(inside(v, p) for v in vests)
        rows.append({"person_id": k, "helmet": has_helmet, "vest": has_vest,
                     "alert": (not has_helmet) or (not has_vest)})
    return rows, int((c == cid["no_helmet"]).sum())

val_imgs = sorted((ROOT / "images" / "val").glob("*"))
report = []
for img, r in zip(val_imgs, best.predict(val_imgs, conf=CONF, imgsz=IMGSZ, nms=True, verbose=False)):
    rows, n_no_helmet = judge(r)
    for row in rows:
        report.append({"image": img.name, **row})
    report.append({"image": img.name, "person_id": -1, "helmet": None, "vest": None, "alert": n_no_helmet > 0,
                   "method_A_no_helmet_boxes": n_no_helmet})
rep = pd.DataFrame(report)
persons = rep[rep["person_id"] >= 0]
print(f"검증 사진 {len(val_imgs)}장 · 감지된 사람 {len(persons)}명")
print(f"방법 B: 헬멧 없음 {int((~persons['helmet'].astype(bool)).sum())}명 · 조끼 없음 {int((~persons['vest'].astype(bool)).sum())}명")
print(f"방법 A: no_helmet 박스가 있는 사진 {int(rep['method_A_no_helmet_boxes'].fillna(0).gt(0).sum())}장")
persons.to_csv(OUT / "violation_report.csv", index=False)
persons.head(10)

In [ ]:
# 경고가 난 사진 한 장을 판정 결과와 함께 그리기
def draw_judgement(img_path):
    r = best.predict(img_path, conf=CONF, imgsz=IMGSZ, nms=True, verbose=False)[0]
    rows, _ = judge(r)
    im = Image.open(img_path).convert("RGB")
    fig, ax = plt.subplots(figsize=(8, 8)); ax.imshow(im)
    b = r.boxes.xyxy.cpu().numpy(); c = r.boxes.cls.cpu().numpy().astype(int)
    for (x1, y1, x2, y2), cc in zip(b, c):
        if NAMES[cc] in ("helmet", "vest"):
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=COLORS[NAMES[cc]], lw=2))
    for row, p in zip(rows, b[c == cid["Person"]]):
        color = "red" if row["alert"] else "lime"
        ax.add_patch(patches.Rectangle((p[0], p[1]), p[2] - p[0], p[3] - p[1], fill=False, color=color, lw=3))
        msg = "OK" if not row["alert"] else ("NO HELMET " if not row["helmet"] else "") + ("NO VEST" if not row["vest"] else "")
        ax.text(p[0], p[1] - 6, msg, color="white", backgroundcolor=color, fontsize=10)
    ax.axis("off"); return fig

alert_imgs = persons.loc[persons["alert"], "image"].unique().tolist()
fig = draw_judgement(ROOT / "images" / "val" / (alert_imgs[0] if alert_imgs else val_imgs[0].name))
fig.savefig(OUT / "alert_example.jpg", dpi=110, bbox_inches="tight"); plt.show()

**질문**: 방법 A와 B가 서로 다른 판정을 내린 사진이 있나요? 현장에 설치한다면 어느 쪽을 쓰고, 그 이유를 한 문장으로 쓸 수 있나요? (이 문장이 그대로 포트폴리오의 "설계 판단"이 됩니다.)

## 7장. 실패 사례 보기

예측과 정답을 IoU 0.5 기준으로 짝지어 **헛탐지(FP)**와 **놓침(FN)**이 많은 사진을 찾고, 정답(초록)과 예측(빨강)을 겹쳐 봅니다.

In [ ]:
def iou(a, b):
    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    return inter / ((a[2]-a[0]) * (a[3]-a[1]) + (b[2]-b[0]) * (b[3]-b[1]) - inter + 1e-9)

def gt_boxes(img_path, W, H):
    out = []
    p = ROOT / "labels" / "val" / f"{img_path.stem}.txt"
    for line in p.read_text().splitlines() if p.exists() else []:
        if line.strip():
            c, cx, cy, w, h = map(float, line.split()[:5])
            out.append((int(c), ((cx-w/2)*W, (cy-h/2)*H, (cx+w/2)*W, (cy+h/2)*H)))
    return out

errs = []
for img, r in zip(val_imgs, best.predict(val_imgs, conf=CONF, imgsz=IMGSZ, nms=True, verbose=False)):
    H, W = r.orig_shape
    gts = gt_boxes(img, W, H); used = set(); tp = 0
    preds = list(zip(r.boxes.cls.cpu().numpy().astype(int), r.boxes.xyxy.cpu().numpy()))
    for pc, pb in preds:
        j = next((k for k, (gc, gb) in enumerate(gts) if k not in used and gc == pc and iou(pb, gb) >= 0.5), None)
        if j is not None: used.add(j); tp += 1
    errs.append({"image": img.name, "TP": tp, "FP": len(preds) - tp, "FN": len(gts) - tp})
err = pd.DataFrame(errs).assign(errors=lambda d: d.FP + d.FN).sort_values("errors", ascending=False)
print("전체:", err[["TP", "FP", "FN"]].sum().to_dict())
err.head(5)

In [ ]:
worst = err.head(2)["image"].tolist()
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, name in zip(axes, worst):
    p = ROOT / "images" / "val" / name
    im = Image.open(p).convert("RGB"); W, H = im.size; ax.imshow(im)
    for c, (x1, y1, x2, y2) in gt_boxes(p, W, H):
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color="lime", lw=2))
    r = best.predict(p, conf=CONF, imgsz=IMGSZ, nms=True, verbose=False)[0]
    for (x1, y1, x2, y2), c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int)):
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color="red", lw=2, ls="--"))
        ax.text(x1, y1 - 3, NAMES[c], color="red", fontsize=8)
    ax.set_title(f"{name}  green=label, red=prediction"); ax.axis("off")
fig.savefig(OUT / "failure_cases.jpg", dpi=110, bbox_inches="tight"); plt.show()

**실패 원인을 세 갈래로 나눠 보세요** — 포트폴리오의 "한계와 개선 방향" 칸이 됩니다.
1. **작아서**: 멀리 있는 사람의 장갑·고글 → 해상도를 올리거나(imgsz 1024) 가까운 사진을 더 모은다
2. **비슷해서**: 모자를 헬멧으로, 반사 조끼 아닌 밝은 옷을 조끼로 → 헷갈리는 사진을 더 넣는다
3. **라벨이 이상해서**: 정답이 빠졌거나 `none`처럼 기준이 흔들림 → 라벨 기준서를 정하고 다시 단다

## 8장. 포트폴리오 한 장 만들기

지금까지의 숫자와 그림을 모아 포트폴리오 초안을 자동으로 씁니다. 괄호 안은 직접 채우세요.

In [ ]:
# 예측 결과 4장을 한 장으로
sample = val_imgs[:4]
tiles = [Image.fromarray(r.plot()[..., ::-1]).resize((480, 480)) for r in best.predict(sample, conf=CONF, imgsz=IMGSZ, nms=True, verbose=False)]
grid = Image.new("RGB", (960, 960))
for k, t in enumerate(tiles): grid.paste(t, ((k % 2) * 480, (k // 2) * 480))
grid.save(OUT / "sample_predictions.jpg"); grid.resize((640, 640))

In [ ]:
weak = per_cls.head(3)["class"].tolist()
n_alert = int(persons["alert"].sum())
md = f'''# 건설현장 안전장비 미착용 감지 (YOLO26)

## 문제
건설현장 CCTV 사진에서 헬멧·조끼 미착용자를 자동으로 찾아 관리자에게 알린다.

## 데이터
- Ultralytics Construction-PPE (AGPL-3.0) · 학습 {len(list((ROOT/"images"/"train").glob("*")))}장 / 검증 {len(val_imgs)}장 · 11클래스
- 클래스 불균형: Person {int(counts["Person"])}개 vs no_boots {int(counts["no_boots"])}개
- 라벨 이슈: `none` 클래스의 정의가 없고 상체·얼굴에 섞여 쓰임 → (내 해석: ______)

## 모델
YOLO26n 전이학습 · {EPOCHS} epochs · imgsz {IMGSZ} · 기본 증강

## 결과
- mAP50 {m.box.map50:.3f} / mAP50-95 {m.box.map:.3f} (검증 세트)
- 약한 클래스: {", ".join(weak)} → 데이터 수·물체 크기와 관련 (ap_vs_data.png)
- 판정 로직(사람 박스 위쪽 1/3 안의 헬멧 + 사람 박스 안의 조끼): 검증 사진에서 경고 {n_alert}건

## 설계 판단
- `no_vest` 클래스가 없어 "사람 + 장비 조합" 규칙으로 판정 (방법 B)
- (방법 A와 B 중 ______를 택한 이유: ______)

## 한계와 개선
- (실패 사례 failure_cases.jpg에서 찾은 원인: ______)
- 다음 단계: 작은 장비용 해상도 상향, 헷갈리는 사진 추가 수집, 라벨 기준서 작성, 영상 추적으로 사람당 1회 경고

## 결과물
sample_predictions.jpg · alert_example.jpg · failure_cases.jpg · metrics_by_class.csv · violation_report.csv
'''
(OUT / "portfolio_ppe.md").write_text(md, encoding="utf-8")
print(md)

In [ ]:
# 결과물 폴더를 zip으로 묶어 내려받기 (Colab)
import shutil
shutil.copy(RUN / "weights" / "best.pt", OUT / "best.pt")
shutil.make_archive("portfolio_ppe", "zip", OUT)
print("만들어진 파일:", sorted(p.name for p in OUT.iterdir()))
try:
    from google.colab import files
    files.download("portfolio_ppe.zip")
except ImportError:
    print("Colab이 아닙니다. portfolio_ppe.zip 을 직접 보관하세요.")

## 정리

- 모델 출력(박스 목록)에 **판정 로직**을 붙여야 "미착용 경고"라는 서비스가 된다
- mAP 하나보다 **클래스별 AP**를 본다 — 데이터가 적고 작은 클래스가 약하다
- 공개 데이터도 라벨을 **직접 열어 본다** — `none`처럼 정의가 흔들리는 클래스가 있다
- 포트폴리오에는 숫자와 함께 **설계 판단**과 **한계**를 쓴다